<div class="blog-language-switch" role="group" aria-label="Article language"><span aria-current="page">English</span><a href="/ipynb/zh-CN/Computer-Science/Computer-Organization/10-io-interrupts-dma-storage.html" lang="zh-CN" hreflang="zh-CN">中文</a></div>

[Back to Computer Organization and Architecture guideline](Computer-Organization.html)

## **I/O, Interrupts, DMA, and Storage** {#io-interrupts-dma-and-storage}

Chapter 09 ended when a storage-backed page fault needed external data. This chapter follows that data beyond the processor-memory boundary. A request moves through software, controller registers and queues, an interconnect, a device-specific engine, and finally a completion path that allows the waiting computation to continue.

Three paths should be kept distinct throughout the chapter:

- the **control path** tells a device what operation to perform;
- the **data path** moves the payload between the device and memory;
- the **completion path** reports success, failure, or partial progress.

A modern storage request may use MMIO for control, DMA for payload, and an MSI-X interrupt or polling for completion. Those mechanisms cooperate; they are not mutually exclusive alternatives.

### **The Role of I/O in a Computer System** {#the-role-of-io-in-a-computer-system}

The processor executes instructions against registers and memory with tightly specified ordering. Peripheral devices operate on very different scales. A keyboard produces sparse events, a display consumes a continuous stream, a network interface receives packets from an unrelated clock, and storage may complete commands out of order after internal error correction. The I/O system reconciles these differences.

::: {.diagram-scroll .wide-diagram}
![CPU cores and memory communicate through a system interconnect with controllers that hide storage, network, display, and input-device physics.](assets/io-system-boundary.svg){fig-align="center"}
:::

A useful abstraction boundary places a **device controller** between the host and the physical device. The host observes registers, queues, buffers, commands, and completion events. The controller converts those operations into device-specific timing, protocols, and error handling. A **device driver** is privileged software that knows this controller interface and presents a safer operating-system abstraction to applications.

| Difference to bridge | Processor-memory side | Device side | Architectural response |
|---|---|---|---|
| timing | instruction-scale, synchronous state | asynchronous external events | status, polling, interrupts, queues |
| speed | nanosecond-scale execution and caches | microseconds to seconds | buffering, batching, overlap |
| granularity | bytes, words, cache blocks | packets, sectors, pages, frames | controllers and protocol descriptors |
| representation | load/store addresses | device-specific commands and signals | registers, command sets, drivers |
| failure | precise architectural exceptions | retries, media errors, disconnects | status codes, timeouts, recovery |
| trust | protected process address spaces | independent bus master | IOMMU, permissions, validation |

The operating system also multiplexes devices among processes, schedules requests, enforces permissions, and turns asynchronous completion into blocking or non-blocking APIs. Hardware provides the mechanisms; software decides policy.

### **Device Controllers and System Interconnects** {#device-controllers-and-system-interconnects}

A device controller has two faces. Its host-visible face exposes configuration space, MMIO registers, command and completion queues, and interrupt capabilities. Its internal face contains state machines or firmware, protocol engines, buffers, error correction, and device-specific scheduling. This separation lets a stable programming interface survive substantial changes in implementation.

::: {.diagram-scroll .wide-diagram}
![A driver reaches a controller through an interconnect; the controller separates host-visible registers and queues from buffers, control logic, and device protocol engines.](assets/device-controller-interconnect.svg){fig-align="center"}
:::

The **system interconnect** transports address, data, and control transactions. An on-chip peripheral bus may connect timers and UARTs, while PCI Express connects high-throughput endpoints. Important properties include bandwidth, arbitration, ordering, error reporting, address routing, hot-plug behavior, and whether a device may initiate memory transactions as a bus master.

Simple controllers may accept one command at a time through control and data registers. High-performance controllers expose memory-resident rings so many requests can remain outstanding. A queue decouples command submission from execution: the host advances a producer position, the controller consumes entries at its own pace, and completion records identify which commands finished.

<details>
<summary>Python model: a controller with command and status registers</summary>

```python
from collections import deque


class DeviceController:
    def __init__(self):
        self.commands = deque()
        self.completed = deque()
        self.busy = False

    def submit(self, command):
        # A real driver would publish a descriptor and notify a doorbell.
        self.commands.append(command)

    def tick(self):
        # The controller advances independently of the CPU instruction stream.
        if self.commands:
            self.busy = True
            command = self.commands.popleft()
            self.completed.append({'id': command['id'], 'status': 'success'})
            self.busy = False

    def read_completion(self):
        return self.completed.popleft() if self.completed else None


controller = DeviceController()
controller.submit({'id': 17, 'operation': 'read', 'lba': 4096})
assert controller.read_completion() is None
controller.tick()
assert controller.read_completion() == {'id': 17, 'status': 'success'}
print('controller returned command 17')
```

</details>

Controller state is concurrent state. Drivers must handle timeouts, stale completions, device reset, queue wraparound, and commands that complete in a different order from submission.

### **Programmed I/O and Memory-Mapped I/O** {#programmed-io-and-memory-mapped-io}

Two independent design questions are often confused:

- **Memory-mapped I/O (MMIO)** answers *how device registers are addressed*. Reserved physical-address ranges are routed to devices, so ordinary load/store instructions access control, status, data, or doorbell registers.
- **Programmed I/O (PIO)** answers *who moves each payload unit*. The CPU explicitly reads or writes every byte or word through a device data register or FIFO.

A driver can therefore use MMIO to configure a DMA transfer. Conversely, an architecture with a separate I/O-port address space can perform programmed I/O using special port instructions rather than MMIO.

::: {.diagram-scroll .wide-diagram}
![An address decoder routes RAM ranges to memory and MMIO ranges to device control, status, data, and doorbell registers; programmed I/O moves each data unit through the CPU.](assets/mmio-programmed-io.svg){fig-align="center"}
:::

Device registers are not ordinary variables. A read may clear an interrupt-pending bit, a write may start irreversible work, a posted write may be buffered before reaching the device, and repeated reads may observe changing hardware state. Drivers therefore use architecture-specific MMIO accessors, prevent compiler removal or reordering, and place memory barriers where the device protocol requires them. MMIO pages are normally mapped with device-appropriate cache attributes rather than ordinary write-back caching.

| Mechanism | Strength | Limitation | Typical use |
|---|---|---|---|
| MMIO register access | reuses load/store address translation and protection | side effects and ordering differ from RAM | control, status, queue doorbells |
| separate I/O ports | keeps device space distinct | special instructions and limited portability | legacy x86 peripherals |
| programmed payload transfer | minimal setup for a few bytes | CPU executes an operation for every data unit | UART byte, tiny FIFO, early boot |
| DMA payload transfer | bulk movement without per-word CPU copying | mapping and synchronization setup | storage, network, audio, display |

<details>
<summary>Python model: MMIO reads and writes with register side effects</summary>

```python
CONTROL, STATUS, DATA = 0x00, 0x04, 0x08
READY, INTERRUPT_PENDING = 0b01, 0b10


class MMIODevice:
    def __init__(self):
        self.status = READY
        self.fifo = []

    def write32(self, offset, value):
        if offset == CONTROL and value == 1:
            self.status &= ~READY  # command starts
        elif offset == DATA:
            self.fifo.append(value & 0xFF)  # CPU moves one byte
        else:
            raise ValueError('unsupported write')

    def read32(self, offset):
        if offset != STATUS:
            raise ValueError('unsupported read')
        value = self.status
        # Model a read-to-acknowledge interrupt bit.
        self.status &= ~INTERRUPT_PENDING
        return value

    def complete(self):
        self.status |= READY | INTERRUPT_PENDING


device = MMIODevice()
for byte in b'IO':
    device.write32(DATA, byte)
device.write32(CONTROL, 1)
device.complete()
first_status = device.read32(STATUS)
second_status = device.read32(STATUS)
assert first_status & INTERRUPT_PENDING
assert not second_status & INTERRUPT_PENDING
assert bytes(device.fifo) == b'IO'
print(bin(first_status), bytes(device.fifo))
```

</details>

### **Polling and Interrupt-Driven I/O** {#polling-and-interrupt-driven-io}

**Polling** repeatedly examines device or queue state until work appears. It is simple and can detect completion with tightly bounded latency, but every unsuccessful check consumes execution bandwidth, interconnect traffic, and often cache capacity. **Interrupt-driven I/O** lets the CPU perform other work or enter a low-power wait; the device signals when service is needed, at the cost of interrupt delivery, privilege entry, handler execution, and return.

::: {.diagram-scroll .wide-diagram}
![Polling checks status repeatedly and discovers completion at the next interval, while interrupt-driven I/O waits efficiently but pays entry, handler, and return overhead.](assets/polling-vs-interrupt.svg){fig-align="center"}
:::

If polling occurs every $\Delta$ seconds and completion time is uniformly distributed within an interval, average detection delay is approximately

$$
E[T_{detect,poll}]\approx\frac{\Delta}{2},
$$

with worst-case delay just under $\Delta$. Here $\Delta$ is the polling interval. Shortening it lowers detection delay but increases the number of checks per second to roughly $1/\Delta$.

A rough CPU-cost comparison over interval $T$ is

$$
C_{poll}\approx\frac{T}{\Delta}c_p,\qquad C_{interrupt}\approx N_ec_i,
$$

where $c_p$ is cost per poll, $N_e$ is number of completion events, and $c_i$ is interrupt entry, handling, and return cost per event. This model omits batching and cache effects but explains the trend: sparse events favor interrupts, while a continuously non-empty high-rate queue can favor polling.

Practical systems combine both. Interrupt coalescing reports several completions together. Adaptive schemes begin with an interrupt, poll while a queue remains busy, and return to interrupts when traffic becomes sparse. The goal is not ideological purity; it is to avoid paying an expensive transition for every tiny event without burning a core on an idle queue.

<details>
<summary>Python example: estimate polling checks and detection delay</summary>

```python
import math


def polling_summary(completion_times, interval):
    detected_at = [math.ceil(t / interval) * interval for t in completion_times]
    delays = [detected - actual for detected, actual in zip(detected_at, completion_times)]
    # Continuous polling through the final detected completion.
    checks = int(max(detected_at) / interval)
    return checks, sum(delays) / len(delays), delays


events = [2.2, 5.7, 9.1]
checks, average_delay, delays = polling_summary(events, interval=1.0)
interrupt_entries = len(events)

assert checks == 10
assert [round(delay, 1) for delay in delays] == [0.8, 0.3, 0.9]
assert interrupt_entries == 3
print(f'poll checks={checks}, average detection delay={average_delay:.2f}')
print(f'interrupt entries={interrupt_entries}')
```

</details>

### **Exceptions and Interrupt Handling** {#exceptions-and-interrupt-handling}

An **exception** is a controlled transfer caused synchronously by the current instruction, such as an illegal opcode, system call, or page fault. An **interrupt** arrives asynchronously from outside the current instruction stream, such as a timer expiry or device completion. ISA terminology varies, but the timing distinction is durable: the exception is explained by the instruction; the interrupt is explained by external pending state.

::: {.diagram-scroll .wide-diagram}
![Synchronous exceptions and asynchronous interrupts converge on architectural state capture, vector selection, privileged handling, acknowledgement, and return.](assets/exception-interrupt-entry.svg){fig-align="center"}
:::

A typical entry sequence records a cause, exception program counter, and status; changes privilege; disables or masks selected interrupts; and jumps through a vector. Software saves any additional registers it will modify, identifies the source, performs urgent service, acknowledges the event, and uses an exception-return instruction to restore prior state. Precise exceptions ensure all older instructions appear complete and no younger instruction appears to have changed architectural state.

An interrupt controller collects sources, applies enable and priority state, and targets a processor context. A level-triggered source remains asserted until its condition is removed; an edge-triggered source records a transition. Message-signaled interrupts encode an interrupt as a special memory write, which fits naturally into packetized interconnects such as PCIe.

The handler should usually do only time-critical work: acknowledge the source, capture completion state, and schedule deferred processing. Long handlers increase interrupt latency for other sources and disrupt the interrupted workload. For claim/complete controllers, claiming identifies the pending source and completion tells the controller that the gateway may forward a new event.

<details>
<summary>Python model: priority, claim, and completion in an interrupt controller</summary>

```python
class InterruptController:
    def __init__(self, priorities):
        self.priorities = priorities
        self.pending = set()
        self.in_service = set()

    def raise_interrupt(self, source):
        self.pending.add(source)

    def claim(self, enabled):
        candidates = self.pending & set(enabled)
        if not candidates:
            return None
        source = max(candidates, key=lambda item: (self.priorities[item], -item))
        self.pending.remove(source)
        self.in_service.add(source)
        return source

    def complete(self, source):
        if source not in self.in_service:
            raise ValueError('source was not claimed')
        self.in_service.remove(source)


controller = InterruptController({1: 2, 2: 7, 3: 4})
for source in [1, 2, 3]:
    controller.raise_interrupt(source)
assert controller.claim(enabled=[1, 2, 3]) == 2
controller.complete(2)
assert controller.claim(enabled=[1, 3]) == 3
print('highest enabled priority is serviced first')
```

</details>

### **Direct Memory Access** {#direct-memory-access}

**Direct memory access (DMA)** lets a controller initiate memory transactions so the CPU does not execute one load and one store for every payload word. The CPU still prepares the transfer: it allocates or pins buffers, obtains device-visible addresses, creates descriptors, publishes them in the required order, notifies the controller, and handles completion and errors. DMA removes repeated copying, not all software involvement.

::: {.diagram-scroll .wide-diagram}
![The CPU maps buffers, writes descriptors, issues a memory barrier, and rings a doorbell; the DMA engine fetches descriptors, transfers data through the fabric and IOMMU, then records completion.](assets/dma-descriptor-transfer.svg){fig-align="center"}
:::

A descriptor commonly includes device address, memory address, length, direction, command identifier, and flags. **Scatter/gather DMA** chains several non-contiguous memory regions into one logical transfer, avoiding a preliminary CPU copy into a physically contiguous buffer.

An IOMMU translates device-visible I/O virtual addresses and limits which physical pages a device may access. This is both an addressing service and a security boundary: a buggy or malicious device should not be able to overwrite arbitrary kernel memory. Buffer lifetime must extend until DMA completes, and page mappings cannot be removed while the device still uses them.

Ordering is essential. Descriptor stores must become visible before the MMIO doorbell that announces them. On a non-coherent system, software may need to clean CPU caches before a device reads memory and invalidate caches after a device writes memory. Even on coherent systems, memory barriers may be required to order payload, descriptor, and completion observations.

For $N$ payload words, a rough CPU-work model is

$$
C_{PIO}=Nc_w,\qquad C_{DMA}=c_s+Kc_d+c_c.
$$

$c_w$ is CPU cost per programmed-I/O word, $c_s$ is fixed DMA setup cost, $K$ is descriptor count, $c_d$ is cost to build one descriptor, and $c_c$ is completion cost. DMA wins when the saved per-word work exceeds setup and synchronization overhead; very small transfers may remain cheaper with programmed I/O.

<details>
<summary>Python model: scatter/gather DMA copies two source segments</summary>

```python
from dataclasses import dataclass


@dataclass
class Descriptor:
    source_offset: int
    destination_offset: int
    length: int


def execute_dma(source, destination, descriptors):
    completed = []
    for index, descriptor in enumerate(descriptors):
        start = descriptor.source_offset
        end = start + descriptor.length
        target = descriptor.destination_offset
        destination[target:target + descriptor.length] = source[start:end]
        completed.append(index)
    return completed


source = bytearray(b'0123456789ABCDEF')
destination = bytearray(b'.' * 16)
ring = [Descriptor(0, 0, 4), Descriptor(8, 4, 8)]
completed = execute_dma(source, destination, ring)

assert completed == [0, 1]
assert destination == bytearray(b'012389ABCDEF....')
print(destination.decode())
```

</details>

### **Persistent Storage** {#persistent-storage}

Persistent storage retains selected data across processor reset and loss of ordinary power. The host usually sees a logical block interface, not raw magnetic domains or individual flash cells. Filesystems and databases build names, allocation, metadata, ordering, and recovery on top of those logical blocks.

Persistence is not the same as a completed CPU store. Data may exist temporarily in a CPU cache, page cache, controller buffer, or volatile drive cache. A durability protocol must define when earlier writes are stable and use mechanisms such as flush, force-unit-access, barriers, journaling, or copy-on-write metadata. Power-loss protection can make some controller caches effectively persistent, but software must follow the documented contract.

| Layer | Unit commonly exposed | Volatile state that may intervene |
|---|---|---|
| application/filesystem | files and logical offsets | language buffers, page cache, metadata cache |
| block layer/driver | logical block requests | request queues and merging |
| device controller | commands and logical block addresses | command/data cache, mapping metadata |
| HDD/SSD media | sectors or NAND pages/erase blocks | device-specific internal pipelines |

The next sections compare two very different media implementations behind a similar block abstraction.

#### **Hard Disk Drives** {#hard-disk-drives}

A **hard disk drive (HDD)** stores bits magnetically on rotating platters. An actuator moves a read/write head radially to a track; platter rotation brings the requested sector under the head; electronics then transfer and decode the data. Because motion dominates many small random requests, physical locality matters even when the interface exposes simple logical block addresses.

::: {.diagram-scroll .wide-diagram}
![A hard-disk head seeks to a track, waits for platter rotation to place the target sector under the head, and then transfers bytes; the latency model separates each component.](assets/hdd-access-components.svg){fig-align="center"}
:::

A useful access-time decomposition is

$$
T_{access}=T_{queue}+T_{ctrl}+T_{seek}+T_{rot}+T_{xfer}.
$$

$T_{queue}$ is time waiting behind other requests, $T_{ctrl}$ is command and controller processing, $T_{seek}$ moves the actuator to the target track, $T_{rot}$ waits for the sector to rotate under the head, and $T_{xfer}$ transfers the requested bytes. These components are not fixed constants; they depend on current head position, request order, block size, and drive behavior.

For rotation rate $r$ revolutions per minute, one revolution takes $60/r$ seconds. Assuming a uniformly random angular position, average rotational delay is half a revolution:

$$
E[T_{rot}]=\frac{30}{r}\text{ seconds}=\frac{30{,}000}{r}\text{ milliseconds}.
$$

At 7,200 RPM this is about 4.17 ms before seek and transfer time. Sequential requests amortize positioning and can stream efficiently; random small I/O repeatedly pays positioning overhead. Request schedulers may reorder independent operations to reduce seek distance, but must balance throughput against fairness and latency deadlines.

<details>
<summary>Python example: estimate HDD access-time components</summary>

```python
def hdd_access_ms(rpm, seek_ms, transfer_bytes, media_mib_per_s, queue_ms=0, controller_ms=0.1):
    average_rotation_ms = 30_000 / rpm
    transfer_ms = transfer_bytes / (media_mib_per_s * 1024**2) * 1000
    total = queue_ms + controller_ms + seek_ms + average_rotation_ms + transfer_ms
    return {
        'rotation_ms': average_rotation_ms,
        'transfer_ms': transfer_ms,
        'total_ms': total,
    }


estimate = hdd_access_ms(
    rpm=7200,
    seek_ms=8.0,
    transfer_bytes=4 * 1024,
    media_mib_per_s=200,
)
assert round(estimate['rotation_ms'], 2) == 4.17
assert estimate['transfer_ms'] < 0.1
assert estimate['total_ms'] > 12
print({key: round(value, 3) for key, value in estimate.items()})
```

</details>

#### **Solid-State Drives** {#solid-state-drives}

A **solid-state drive (SSD)** stores data in non-volatile flash without moving heads or platters. Lack of mechanical positioning greatly reduces random-access latency, but NAND flash introduces a different asymmetry: data is read and programmed in pages, while reuse requires erasing a much larger erase block. A programmed page cannot simply be overwritten in place.

::: {.diagram-scroll .wide-diagram}
![An SSD controller translates host logical block addresses through an FTL, writes an update to a free NAND page, marks the old page stale, and later reclaims erase blocks through garbage collection.](assets/ssd-ftl-nand.svg){fig-align="center"}
:::

The **flash translation layer (FTL)** maps host logical block addresses to current physical flash pages. An update is written to a fresh page, the mapping changes, and the old page becomes stale. Garbage collection copies remaining valid pages from a partially stale block and erases the whole block for reuse. The extra internal traffic creates **write amplification**:

$$
WA=\frac{B_{NAND,written}}{B_{host,written}},
$$

where $B_{NAND,written}$ is total bytes programmed internally, including copied data, and $B_{host,written}$ is bytes requested by the host. $WA=1$ is ideal; values above one consume bandwidth and program/erase endurance.

Wear leveling distributes erases so a small set of blocks does not fail early. Over-provisioned capacity gives the controller free space for remapping and garbage collection. Error-correcting codes, retry logic, bad-block management, multiple NAND channels, and controller DRAM or host-memory buffers all affect observable performance. TRIM/deallocation tells the SSD that selected logical blocks no longer contain useful host data, allowing their physical pages to be reclaimed without copying.

<details>
<summary>Python model: out-of-place updates in a toy FTL</summary>

```python
class ToyFTL:
    def __init__(self, physical_pages):
        self.pages = [None] * physical_pages
        self.valid = [False] * physical_pages
        self.mapping = {}

    def write(self, lba, value):
        # NAND update is placed in a free physical page.
        try:
            new_page = next(i for i, page in enumerate(self.pages) if page is None)
        except StopIteration as error:
            raise RuntimeError('garbage collection required') from error

        if lba in self.mapping:
            self.valid[self.mapping[lba]] = False  # old version becomes stale
        self.pages[new_page] = (lba, value)
        self.valid[new_page] = True
        self.mapping[lba] = new_page
        return new_page

    def read(self, lba):
        page = self.mapping[lba]
        return self.pages[page][1]


ftl = ToyFTL(physical_pages=6)
old_page = ftl.write(42, 'old')
ftl.write(7, 'other')
new_page = ftl.write(42, 'new')

assert old_page != new_page
assert not ftl.valid[old_page]
assert ftl.valid[new_page]
assert ftl.read(42) == 'new'
print(f'LBA 42 moved from physical page {old_page} to {new_page}')
```

</details>

Compared with an HDD, an SSD removes seek and rotational delay and can exploit many internal channels, but its latency still varies with queueing, garbage collection, thermal throttling, error retries, cache state, and write history.

#### **Storage Performance** {#storage-performance}

Storage performance cannot be summarized by one number. **Latency** measures time from submission to completion for one request. **IOPS** counts completed operations per second. **Bandwidth** counts completed bytes per second. **Queue depth** counts outstanding requests. **Tail latency** reports slow percentiles that averages hide.

::: {.diagram-scroll .wide-diagram}
![Several outstanding requests feed a device scheduler; the diagram distinguishes latency, IOPS, bandwidth, queue depth, tail latency, and Little's Law.](assets/storage-performance-queue.svg){fig-align="center"}
:::

For an average request size $S$ bytes and completion rate $\lambda$ operations per second,

$$
Bandwidth=\lambda S.
$$

Here $\lambda$ is IOPS and $S$ is average transferred bytes per operation. The same IOPS can therefore mean very different bandwidth for 4 KiB random reads and 1 MiB sequential reads.

Little's Law relates steady-state average concurrency $N$, completion rate $\lambda$, and average response time $W$:

$$
N=\lambda W.
$$

$N$ is average in-flight requests, $\lambda$ is requests completed per unit time, and $W$ uses the same time unit. Rearranging gives $\lambda=N/W$, an idealized upper estimate when enough parallel device service exists. Increasing queue depth helps until internal parallelism is occupied; after saturation, new requests mostly wait, so throughput flattens while latency grows.

Measurements must state workload shape: read/write ratio, random/sequential addresses, request size, queue depth, duration, warm-up, cache policy, durability requirements, data compressibility, and reported percentile. Comparing a cached sequential benchmark with durable random writes is not meaningful.

<details>
<summary>Python example: connect queue depth, latency, IOPS, and bandwidth</summary>

```python
def ideal_storage_rate(queue_depth, average_latency_ms, request_bytes, device_iops_limit=None):
    latency_seconds = average_latency_ms / 1000
    little_law_iops = queue_depth / latency_seconds
    iops = min(little_law_iops, device_iops_limit) if device_iops_limit else little_law_iops
    bandwidth_mib_s = iops * request_bytes / 1024**2
    return iops, bandwidth_mib_s


iops, bandwidth = ideal_storage_rate(
    queue_depth=32,
    average_latency_ms=0.2,
    request_bytes=4096,
    device_iops_limit=120_000,
)
assert iops == 120_000  # device saturates below the unconstrained 160k estimate
assert round(bandwidth, 2) == 468.75
print(f'{iops:,.0f} IOPS, {bandwidth:.2f} MiB/s')
```

</details>

### **RAID and Storage Reliability** {#raid-and-storage-reliability}

**RAID** combines several drives into one logical storage system to change capacity, performance, and tolerance of selected drive failures. Striping distributes blocks, mirroring stores copies, and parity stores enough information to reconstruct missing data. The chosen layout determines usable capacity and the number and pattern of failures that can be survived.

::: {.diagram-scroll .wide-diagram}
![Four-drive examples compare RAID 0 striping, RAID 1 mirrored pairs, RAID 5 distributed parity, and RAID 10 striped mirrors.](assets/raid-layout-comparison.svg){fig-align="center"}
:::

For $N$ equal drives of size $S$, common usable-capacity approximations are

$$
C_{RAID0}=NS,\quad C_{RAID5}=(N-1)S,\quad C_{RAID6}=(N-2)S,\quad C_{RAID10}=\frac{N}{2}S.
$$

$N$ is drive count and $S$ is capacity of the smallest participating drive. RAID 10 assumes an even number of drives arranged as mirrored pairs. RAID 1 capacity depends on mirror grouping; a simple two-drive mirror exposes $S$.

| Level | Minimum drives | Usable capacity | Failure tolerance | Important write behavior |
|---|---:|---:|---|---|
| RAID 0 | 2 | $NS$ | none | parallel data writes, no redundancy |
| RAID 1 | 2 | one copy per mirror group | one or more if each group retains a copy | update every mirror |
| RAID 5 | 3 | $(N-1)S$ | one drive | small writes may read/modify/write data and parity |
| RAID 6 | 4 | $(N-2)S$ | two drives | two parity syndromes increase write work |
| RAID 10 | 4 | $NS/2$ | depends on which drives fail | stripe across mirrored pairs |

For RAID 5, parity can be expressed bytewise as $P=D_0\oplus D_1\oplus D_2$. If $D_1$ is missing, $D_1=P\oplus D_0\oplus D_2$ because XOR is its own inverse. Distributed parity avoids one permanent parity drive, but degraded reads and rebuilds consume surviving-drive bandwidth. Large arrays face long rebuild windows and latent media errors, so practical reliability also uses checksums, scrubbing, hot spares, monitoring, and backups.

<details>
<summary>Python example: reconstruct one missing RAID-5 data block with XOR</summary>

```python
def xor_blocks(*blocks):
    if len({len(block) for block in blocks}) != 1:
        raise ValueError('all blocks must have equal length')
    return bytes(a ^ b ^ c for a, b, c in zip(*blocks)) if len(blocks) == 3 else bytes(
        __import__('functools').reduce(lambda x, y: x ^ y, values)
        for values in zip(*blocks)
    )


data0 = bytes([0x10, 0x22, 0x34, 0x48])
data1 = bytes([0xAB, 0xCD, 0xEF, 0x01])
data2 = bytes([0x55, 0x66, 0x77, 0x88])
parity = xor_blocks(data0, data1, data2)
recovered_data1 = xor_blocks(parity, data0, data2)

assert recovered_data1 == data1
print('recovered:', recovered_data1.hex())
```

</details>

RAID is not a backup. It cannot by itself recover an accidentally deleted file, corrupted application data replicated to every member, stolen hardware, or site-wide disaster. Redundancy improves availability for defined component failures; backup and recovery protect historical data.

### **Tracing an End-to-End I/O Request** {#tracing-an-end-to-end-io-request}

Consider a 16 KiB read from an NVMe SSD when the requested file data is absent from the page cache. This path demonstrates how the chapter's mechanisms compose. A cache hit would end inside memory and skip the device path entirely.

::: {.diagram-scroll .wide-diagram}
![An application read crosses the filesystem, block layer, NVMe submission queue, MMIO doorbell, controller, NAND, DMA destination, completion queue, and interrupt or polling path before the task resumes.](assets/end-to-end-nvme-read.svg){fig-align="center"}
:::

1. The application issues `read()` with a virtual destination buffer and file offset.
2. The filesystem checks the page cache and translates file blocks into a block request.
3. The block layer may merge, split, and schedule the request before passing it to the NVMe driver.
4. The driver pins or maps destination pages for DMA, writes a submission queue entry (SQE), and publishes it with the required memory ordering.
5. An MMIO doorbell write tells the controller that the submission queue tail advanced.
6. The controller DMA-fetches the command, validates it, and maps the logical block address through its FTL.
7. NAND channels read the required pages; controller ECC corrects recoverable errors.
8. The controller DMA-writes payload bytes into the mapped host-memory pages.
9. After data visibility is guaranteed, the controller posts a completion queue entry and signals MSI-X or waits to be polled.
10. The driver consumes the completion, handles errors, unmaps DMA state, and wakes the task; the filesystem returns the requested bytes.

The command and doorbell form the control path. The DMA write forms the data path. The completion queue and interrupt/poll form the completion path. This separation explains why an interrupt does not carry the 16 KiB payload and why DMA does not tell software which application request should be completed without descriptors and command identifiers.

A storage-backed page fault from Chapter 09 joins near step 2: the VM subsystem allocates a frame, asks the filesystem or swap subsystem for the missing page, and restarts the faulting instruction only after the I/O path fills that frame.

<details>
<summary>Python model: build a timestamped control, data, and completion trace</summary>

```python
phases = [
    ('syscall_and_cache_lookup', 'control', 4),
    ('block_and_driver_setup', 'control', 8),
    ('publish_sqe_and_doorbell', 'control', 2),
    ('controller_and_flash_read', 'device', 70),
    ('dma_payload_to_memory', 'data', 6),
    ('post_cqe_and_interrupt', 'completion', 5),
    ('driver_wakeup_and_return', 'completion', 5),
]


def build_trace(items):
    now_us = 0
    trace = []
    for name, path, duration_us in items:
        trace.append({'start_us': now_us, 'end_us': now_us + duration_us, 'path': path, 'name': name})
        now_us += duration_us
    return trace


trace = build_trace(phases)
assert trace[-1]['end_us'] == 100
assert [item['path'] for item in trace].count('data') == 1
assert trace[4]['name'] == 'dma_payload_to_memory'
for item in trace:
    print(f"{item['start_us']:>3}-{item['end_us']:>3} us  {item['path']:<10} {item['name']}")
```

</details>

### **Choosing an I/O Transfer Mechanism** {#choosing-an-io-transfer-mechanism}

The right design chooses control, payload movement, and completion independently. Register-sized commands usually use MMIO. Large payloads usually use DMA. Sparse completions favor interrupts; continuously busy queues may favor polling or an adaptive combination.

::: {.diagram-scroll .wide-diagram}
![A decision flow selects MMIO programmed I/O for small control, DMA for bulk payload, interrupts for sparse completions, and polling or batching for high-rate completions.](assets/io-mechanism-selection.svg){fig-align="center"}
:::

| Workload | Control | Payload | Completion | Why |
|---|---|---|---|---|
| configure a timer | MMIO registers | none/tiny | poll once or interrupt | setup dominates |
| keyboard event | MMIO status | tiny FIFO read | interrupt | events are sparse and unpredictable |
| high-speed network receive | MMIO doorbells | DMA descriptor rings | interrupt then poll batch | many small packets amortize transitions |
| NVMe block I/O | MMIO doorbell | DMA to mapped pages | CQ polling or MSI-X | queues expose parallelism |
| audio playback | MMIO control | cyclic DMA buffer | periodic interrupt | continuous stream with deadlines |
| early boot UART | MMIO/port I/O | programmed byte transfer | polling | simplicity before full interrupt setup |

Selection must also consider failure behavior. Polling needs a timeout rather than an infinite loop. Interrupt handlers must tolerate shared or stale events. DMA needs bounded addresses, valid buffer lifetime, cancellation rules, and reset recovery. Completion must be correlated with the correct command before memory is returned to another owner.

<details>
<summary>Python example: choose control, payload, and completion mechanisms separately</summary>

```python
def choose_io_strategy(payload_bytes, events_per_second, latency_critical=False):
    control = 'MMIO registers/doorbell'
    payload = 'programmed I/O' if payload_bytes <= 16 else 'DMA'

    if events_per_second < 1_000:
        completion = 'interrupt'
    elif events_per_second > 100_000 or latency_critical:
        completion = 'polling or adaptive polling'
    else:
        completion = 'interrupt coalescing'
    return control, payload, completion


assert choose_io_strategy(1, 20) == (
    'MMIO registers/doorbell', 'programmed I/O', 'interrupt'
)
assert choose_io_strategy(16 * 1024, 200_000)[1:] == (
    'DMA', 'polling or adaptive polling'
)
print(choose_io_strategy(4096, 10_000))
```

</details>

The numeric thresholds in this toy policy are illustrative, not hardware rules. Real selection depends on measured setup cost, event rate, power budget, cache behavior, device queueing, and latency targets.

**Chapter summary.** I/O connects the synchronous processor-memory model to devices with different timing, granularity, reliability, and protocols. Controllers expose registers and queues while hiding device-specific execution. MMIO determines how registers are addressed; programmed I/O determines whether the CPU moves each payload unit. Polling trades CPU attention for direct detection, while interrupts trade entry overhead for efficient waiting. Exceptions are synchronous to an instruction; device interrupts are asynchronous and commonly pass through a priority controller or message-signaled path. DMA lets a controller move bulk data between a device and mapped memory, but still requires descriptor setup, ordering, lifetime management, IOMMU protection, and completion handling. HDDs pay seek and rotational delay; SSDs replace mechanics with FTL mapping, out-of-place writes, garbage collection, wear leveling, and flash-channel parallelism. Storage performance must distinguish latency, IOPS, bandwidth, queue depth, and tails. RAID changes placement and selected failure tolerance but does not replace backups. Chapter 11 will extend these ideas to multicore execution, where per-core queues, interrupt routing, shared-memory synchronization, and cache coherence determine how I/O scales across processors.